In [ ]:
# Install pinned dependencies (do NOT install vLLM in this environment)
!pip install transformers accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
import time, threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**enc, max_new_tokens=new_tokens, do_sample=False, streamer=streamer)
    th = threading.Thread(target=model.generate, kwargs=kwargs)

    t0 = time.time()
    th.start()
    stamps = []
    for _ in streamer:
        stamps.append(time.time())
    th.join()

    ttft = stamps[0] - t0
    if len(stamps) > 1:
        gaps = [b - a for a, b in zip(stamps, stamps[1:])]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0
    total = stamps[-1] - t0

    return {
        "ttft_s": round(ttft, 4),
        "tpot_s": round(tpot, 4),
        "total_s": round(total, 4),
        "n_tokens": len(stamps)
    }

# Warm-up call (throwaway to absorb CUDA context init)
measure_stream(prompt_of_len(128), new_tokens=8)

ttft_by_len = {}
for n in [128, 512, 2048]:
    r = measure_stream(prompt_of_len(n))
    ttft_by_len[str(n)] = r["ttft_s"]
    print(f"Context: {n}", r)

Context: 128 {'ttft_s': 0.2647, 'tpot_s': 0.1427, 'total_s': 18.5329, 'n_tokens': 129}
Context: 512 {'ttft_s': 0.1472, 'tpot_s': 0.0853, 'total_s': 11.0599, 'n_tokens': 129}
Context: 2048 {'ttft_s': 0.6948, 'tpot_s': 0.0883, 'total_s': 11.9998, 'n_tokens': 129}


In [ ]:
 import gc
import json

def kv_formula_kb_per_token(layers=28, kv_heads=2, head_dim=128, dbytes=2):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024  # 28.0 KB

def cache_bytes(pkv):
    """Recursively extracts all PyTorch tensors from any cache structure."""
    if pkv is None:
        return 0

    tensors = []

    # If the cache object exposes .layers (newer DynamicCache)
    if hasattr(pkv, "layers"):
        for layer in pkv.layers:
            for attr in ("keys", "values", "key", "value"):
                t = getattr(layer, attr, None)
                if isinstance(t, torch.Tensor):
                    tensors.append(t)

    # Standard DynamicCache with key_cache / value_cache attributes
    elif hasattr(pkv, "key_cache") and hasattr(pkv, "value_cache"):
        for k, v in zip(pkv.key_cache, pkv.value_cache):
            if isinstance(k, torch.Tensor):
                tensors.append(k)
            if isinstance(v, torch.Tensor):
                tensors.append(v)

    # Fallback to legacy tuple/list traversal
    elif isinstance(pkv, (list, tuple)):
        def extract(obj):
            if isinstance(obj, torch.Tensor):
                tensors.append(obj)
            elif isinstance(obj, (list, tuple)):
                for item in obj:
                    extract(item)
        extract(pkv)

    return sum(t.numel() * t.element_size() for t in tensors)


def measure_kv(context: int, new_tokens: int = 256):
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    enc = tok(prompt_of_len(context), return_tensors="pt").to("cuda")
    before = torch.cuda.memory_allocated()
    out = model.generate(
        **enc, max_new_tokens=new_tokens, do_sample=False,
        use_cache=True, return_dict_in_generate=True
    )
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated()
    total_tokens = out.sequences.shape[1]

    return {
        "context": context,
        "total_tokens": int(total_tokens),
        "peak_kb_per_token": round((peak - before) / total_tokens / 1024, 1),
        "kv_kb_per_token": round(cache_bytes(out.past_key_values) / total_tokens / 1024, 1),
    }

formula = kv_formula_kb_per_token()
print("formula KB/token:", formula)

kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]
for r in kv_rows:
    print(r, "  vs formula", formula, "KB/token")

with open("kv_check.json", "w") as f:
    json.dump({
        "formula_kb_per_token": formula,
        "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
        "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"]
    }, f)

formula KB/token: 28.0
{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 81.7, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 258.4, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 484.3, 'kv_kb_per_token': 28.0}   vs formula 28.0 KB/token


In [ ]:
# 24 requests: 18 asking for 32 tokens, 6 asking for 256 tokens
QUEUE = [32, 32, 32, 256] * 6

def static_queue(batch: int, prompt: str = "Explain what an inference server does."):
    t0 = time.time()
    useful = 0
    slots = 0

    for i in range(0, len(QUEUE), batch):
        chunk = QUEUE[i:i + batch]
        n = max(chunk)
        enc = tok([prompt] * len(chunk), return_tensors="pt", padding=True).to("cuda")
        model.generate(**enc, max_new_tokens=n, do_sample=False)
        useful += sum(chunk)
        slots += n * len(chunk)

    dt = time.time() - t0
    return {
        "batch": batch,
        "wall_s": round(dt, 2),
        "tokens_per_s": round(useful / dt, 1),
        "slot_efficiency": round(useful / slots, 3)
    }

batch_rows = {}
for n in [1, 4, 8]:
    r = static_queue(n)
    batch_rows[str(n)] = r["tokens_per_s"]
    print(r)

{'batch': 1, 'wall_s': 71.79, 'tokens_per_s': 29.4, 'slot_efficiency': 1.0}


KeyboardInterrupt: 

In [ ]:
import json
from google.colab import files

baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,
    "tpot_s": measure_stream(prompt_of_len(512))["tpot_s"],
    "batch": {k: v for k, v in batch_rows.items()},
}

with open("baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)

print(json.dumps(baselines, indent=2))

# Download immediately so it survives runtime disconnection
files.download("baselines.json")

In [ ]:
import json, os

KV_FORMULA_KB_PER_TOKEN = 2 * 28 * 2 * 128 * 2 / 1024  # 28.0

class _Stop(Exception):
    pass

def fail(reason: str):
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()

def load_json(path: str):
    if not os.path.exists(path):
        fail(f"{path} not found")
    try:
        with open(path) as fh:
            return json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

def main() -> None:
    b = load_json("baselines.json")

    for key in ("model", "dtype", "ttft_s", "tpot_s", "batch"):
        if key not in b:
            fail(f"baselines.json missing key: {key}")
    if not isinstance(b["ttft_s"], dict) or not b["ttft_s"]:
        fail("ttft_s must be a non-empty object keyed by prompt length")
    if not isinstance(b["batch"], dict):
        fail("batch must be an object keyed by batch size")
    for size in ("1", "4", "8"):
        if size not in b["batch"]:
            fail(f"batch missing size {size}")

    tpot = b["tpot_s"]
    if not isinstance(tpot, (int, float)) or tpot <= 0:
        fail(f"tpot_s not a positive number: {tpot}")
    for plen, ttft in b["ttft_s"].items():
        if not isinstance(ttft, (int, float)) or ttft <= 0:
            fail(f"ttft_s[{plen}] not a positive number: {ttft}")
    if not b["ttft_s"]["2048"] > b["ttft_s"]["128"]:
        fail(f"TTFT did not rise with prompt length (128: {b['ttft_s']['128']}, 2048: {b['ttft_s']['2048']})")

    b1, b8 = b["batch"]["1"], b["batch"]["8"]
    if not (isinstance(b1, (int, float)) and isinstance(b8, (int, float))):
        fail("batch tokens/s values must be numbers")
    if not b8 > b1:
        fail(f"batch-8 throughput ({b8}) not above batch-1 ({b1})")

    kv = load_json("kv_check.json")
    measured = kv.get("measured_kb_per_token")
    if not isinstance(measured, (int, float)) or measured <= 0:
        fail("kv_check.json needs a positive measured_kb_per_token")
    lo, hi = KV_FORMULA_KB_PER_TOKEN / 2, KV_FORMULA_KB_PER_TOKEN * 2
    if not lo <= measured <= hi:
        fail(f"measured KV {measured} KB/token outside 2x of formula {KV_FORMULA_KB_PER_TOKEN}")

    print(f"ttft lengths: {sorted(b['ttft_s'])}, tpot_s: {tpot}")
    print(f"batch tokens/s 1/4/8: {b['batch']['1']}/{b['batch']['4']}/{b['batch']['8']}")
    print(f"KV measured {measured} KB/token vs formula {KV_FORMULA_KB_PER_TOKEN} KB/token")
    print("GREEN CHECK: PASS")

try:
    main()
except _Stop:
    pass

ttft lengths: ['128', '2048', '512'], tpot_s: 0.0431
batch tokens/s 1/4/8: 22.3/38.4/85.3
KV measured 28.0 KB/token vs formula 28.0 KB/token
GREEN CHECK: PASS
